### <span style="color:red">!caution</span>
Attribute Lens visualizations are implemented with `plotly`. Currently github can't render `plotly` figures.

In [6]:
import os
import sys
sys.path.append('..')

import torch
from src import models, data
from src.attributelens.attributelens import Attribute_Lens
import src.attributelens.utils as lens_utils
import numpy as np

In [7]:
import plotly.io as pio

# pio.renderers.default = "notebook_connected"
pio.renderers.default = "iframe_connected"

In [8]:
# LREs are caches for GPT-J. 
device = "cuda"
mt = models.load_model("gptj", device=device, fp16=True)
print(f"dtype: {mt.model.dtype}, device: {mt.model.device}, memory: {mt.model.get_memory_footprint()}")

Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

[transformers] GPTJForCausalLM LOAD REPORT from: EleutherAI/gpt-j-6B
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...27}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...27}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


dtype: torch.float16, device: cuda:0, memory: 12109105600


In [9]:
# downloading cached LREs
# ! gdown --no-check-certificate --folder https://drive.google.com/drive/u/0/folders/1jAxqpACq5-gDbHG3cFhrL8eC65UtPcUM

In [43]:
task='verb'

In [44]:
# word first/last letter
if task =='word':
    prompt = mt.tokenizer.eos_token + " " + "BEAUTIFUL ASK EXPAND FRONT"

if task =='adjective':
    prompt = mt.tokenizer.eos_token + " " + "it's a small house."

# verb past tense
if task=='verb':
    prompt = mt.tokenizer.eos_token + " " + "I think he run away."

## Attribute Lens

In [45]:
from src.operators import LinearRelationOperator

def load_cached_lre(relation_name, path = "../results/LRE_cached"):
    approx = np.load(os.path.join(path, relation_name.replace(" ", "_") + ".npz"), allow_pickle=True)
    approx_dict = {}
    for key,value in approx.items():
        if key in ["h", "z", "weight", "bias"]:
            approx_dict[key] = torch.from_numpy(value).cuda()
        else:
            approx_dict[key] = value.item()
    return LinearRelationOperator(
        mt = mt, 
        weight = approx_dict["weight"],
        bias = approx_dict["bias"],
        h_layer = approx_dict["h_layer"],
        z_layer = approx_dict["z_layer"],
        prompt_template = approx_dict["prompt_template"],
        beta = approx_dict["beta"]
    )

In [46]:
# # Uncomment the block and print `relation_names` to see all the options
# dataset = data.load_dataset()
# relation_names = [r.name for r in dataset.relations]
# relation_names

In [47]:
relation_names = [
    "word first letter",
    "word last letter",
    "adjective antonym",
    "adjective superlative",
    "verb past tense",
    "adjective comparative",
]

In [48]:
if task =='word':
    relation_names=["word first letter","word last letter"]

if task =='adjective':
    relation_names = [
        "adjective antonym",
        "adjective superlative",
        "adjective comparative",
    ]

if task=='verb':
    relation_names=["verb past tense"]

In [49]:
lres = {
    relation_name: load_cached_lre(relation_name = relation_name,path='LRE_cached')
    for relation_name in relation_names
}

In [50]:
import time

lens = Attribute_Lens(mt=mt, top_k=10)

colorscales = ["blues", "oranges", "greens", "reds", "purples", "greys"]

for relation_name, colorscale in zip(relation_names, colorscales[:len(relation_names)]):
    print("----------------------------------------")
    print(relation_name, " -- ", colorscale)
    print("----------------------------------------")
    att_info = lens.apply_attribute_lens(
        prompt=prompt,
        relation_operator=lres[relation_name]
    )
    att_info['subject_range']= (1, att_info['subject_range'][-1]) # ignore the first EOS token
    p = lens_utils.visualize_attribute_lens(
        att_info, layer_skip=2, must_have_layers=[],
        colorscale= colorscale
    )
    p.layout.margin = dict(l=0, r=0, t=0, b=0)
    p.show()
    
    p.write_html(
    f"attribute_lens_{relation_name}.html",
    include_plotlyjs="cdn",  # smaller file; needs internet when opened
    full_html=True,
)
    
    time.sleep(1)

----------------------------------------
verb past tense  --  blues
----------------------------------------


## Logit Lens

In [51]:
prompt

'<|endoftext|> I think he run away.'

In [52]:
lens = Attribute_Lens(mt=mt, top_k=10)
att_info = lens.apply_attribute_lens(
    prompt=prompt,
    relation_operator=None # Will use Identity if set to None. Basically Logit Lens
)
att_info['subject_range']= (1, att_info['subject_range'][-1]) # ignore the first EOS token
# print('prediction:', att_info['nextwords'][-1])
p = lens_utils.visualize_attribute_lens(
    att_info, layer_skip=2, must_have_layers=[],
)
p.layout.margin = dict(l=0, r=0, t=0, b=0)
p.show()
p.write_html(
    f"logit_lens_{task}.html",
    include_plotlyjs="cdn",  # smaller file; needs internet when opened
    full_html=True,
)